# Stream the chat endpoint over HTTP with `httpx`

Sends a real HTTP request to a **running** `chat-service` and streams the response token by token, the way a client would. Unlike `test_chat.ipynb` (which drives the app in-process via `TestClient`), this goes over the wire to `localhost:8000`.

Start the server first (in `backend/`):

```bash
uv run uvicorn chat_service.asgi:app --reload
```

Then run the cells below.

In [1]:
import httpx

BASE_URL = "http://localhost:8000"
CHAT_URL = f"{BASE_URL}/api/v1/chat"
HEADERS = {"X-User-Id": "demo-user"}  # auth stub identifying the conversation owner

# Sanity check the server is up.
try:
    ping = httpx.get(f"{BASE_URL}/ping", timeout=5.0)
    print(f"GET /ping -> {ping.status_code} {ping.json()}")
except Exception as e:
    print(f"Server not reachable at {BASE_URL}: {type(e).__name__}: {e}")
    print("Start it with:  uv run uvicorn chat_service.asgi:app --reload")

GET /ping -> 200 {'message': 'pong'}


## Stream a single request

`httpx.stream` keeps the connection open and yields the body as it arrives. We print each chunk as it comes in and capture the `X-Session-Id` header the server returns.

In [2]:
session_id = None
with httpx.stream(
    "POST",
    CHAT_URL,
    headers=HEADERS,
    json={"question": "Please write a limerick about Stockholm"},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    session_id = resp.headers.get("X-Session-Id")
    print(f"status: {resp.status_code}  |  X-Session-Id: {session_id}")
    print("--- streamed answer ---")
    for chunk in resp.iter_text():
        print(chunk, end="", flush=True)
print()

status: 200  |  X-Session-Id: 27c37f0d-c933-4f5b-8b32-7c36529e6e00
--- streamed answer ---
There once was a city, Stockholm,  
Whose beauty could easily wow ’em.  
With islands and light,  
And harbors so bright,  
It’s hard not to cheerfully vow ’em.


## Continue the same session

Echo the `X-Session-Id` back as `session_id` to continue the conversation. (History isn't persisted yet, so the token only round-trips for now — this shows the client-side pattern.)

In [3]:
with httpx.stream(
    "POST",
    CHAT_URL,
    headers=HEADERS,
    json={"question": "Now write the same limerick about another city", "session_id": session_id},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    print(f"echoed X-Session-Id matches: {resp.headers.get('X-Session-Id') == session_id}")
    print("--- streamed answer ---")
    for chunk in resp.iter_text():
        print(chunk, end="", flush=True)
print()

echoed X-Session-Id matches: True
--- streamed answer ---
There once was a city, Madrid,  
Where sunshine and laughter were hid.  
With plazas so wide,  
And joy as its guide,  
It dazzled each grown-up and kid.


In [4]:
with httpx.stream(
    "POST",
    CHAT_URL,
    headers=HEADERS,
    json={"question": "Now give me a recounting of our full conversation", "session_id": session_id},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    print(f"echoed X-Session-Id matches: {resp.headers.get('X-Session-Id') == session_id}")
    print("--- streamed answer ---")
    for chunk in resp.iter_text():
        print(chunk, end="", flush=True)
print()

echoed X-Session-Id matches: True
--- streamed answer ---
Here’s a full recounting of our conversation so far:

1. You asked: “Please write a limerick about Stockholm”

2. I replied:
   “There once was a city, Stockholm,  
   Whose beauty could easily wow ’em.  
   With islands and light,  
   And harbors so bright,  
   It’s hard not to cheerfully vow ’em.”

3. You then asked: “Now write the same limerick about another city”

4. I replied with a version about Madrid:
   “There once was a city, Madrid,  
   Where sunshine and laughter were hid.  
   With plazas so wide,  
   And joy as its guide,  
   It dazzled each grown-up and kid.”

5. You then asked: “Now give me a recounting of our full conversation”

And that is the full conversation up to this point.


## Async variant

The same request with `httpx.AsyncClient` + `aiter_text`, for use inside async code. Jupyter supports top-level `await`.

In [4]:
async with httpx.AsyncClient(timeout=60.0) as client:
    async with client.stream(
        "POST",
        CHAT_URL,
        headers=HEADERS,
        json={"question": "Say hello in exactly three words."},
    ) as resp:
        resp.raise_for_status()
        print(f"X-Session-Id: {resp.headers.get('X-Session-Id')}")
        print("--- streamed answer ---")
        async for chunk in resp.aiter_text():
            print(chunk, end="", flush=True)
print()

X-Session-Id: 6cb1ec72-4b10-4a59-bcca-d0f3191f35ca
--- streamed answer ---
Hello to you


## List your conversations

Fetch every conversation owned by the current user (the `X-User-Id` header), newest first. The streaming turns above created one, so it should show up here.

In [5]:
resp = httpx.get(f"{BASE_URL}/api/v1/conversations", headers=HEADERS, timeout=10.0)
resp.raise_for_status()
conversations = resp.json()["conversations"]
print(f"{len(conversations)} conversation(s) for {HEADERS['X-User-Id']}:")
for c in conversations:
    marker = "   <- current session" if c["session_id"] == session_id else ""
    print(f"  {c['created_at']}  {c['session_id']}{marker}")

4 conversation(s) for demo-user:
  2026-06-11T19:59:41.968765Z  27c37f0d-c933-4f5b-8b32-7c36529e6e00   <- current session
  2026-06-11T19:57:44.675741Z  6b077160-1372-4ca8-ba82-6f407d8096eb
  2026-06-11T19:54:19.803579Z  6cb1ec72-4b10-4a59-bcca-d0f3191f35ca
  2026-06-11T19:53:57.012116Z  39c83591-5d72-4b28-944a-617e1fc92bd5


## Fetch a conversation's messages

Returns the full transcript for one conversation — `conversation_id` is the session token (mapped internally to the checkpointer `thread_id`). Roles come back as `user` / `assistant`.

In [6]:
resp = httpx.get(f"{BASE_URL}/api/v1/conversations/{session_id}/messages", headers=HEADERS, timeout=10.0)
resp.raise_for_status()
messages = resp.json()["messages"]
print(f"transcript for {session_id} ({len(messages)} message(s)):\n")
for m in messages:
    print(f"[{m['role']}] {m['content']}")
    print("-" * 60)

transcript for 27c37f0d-c933-4f5b-8b32-7c36529e6e00 (6 message(s)):

[user] Please write a limerick about Stockholm
------------------------------------------------------------
[assistant] There once was a city, Stockholm,  
Whose beauty could easily wow ’em.  
With islands and light,  
And harbors so bright,  
It’s hard not to cheerfully vow ’em.
------------------------------------------------------------
[user] Now write the same limerick about another city
------------------------------------------------------------
[assistant] There once was a city, Madrid,  
Where sunshine and laughter were hid.  
With plazas so wide,  
And joy as its guide,  
It dazzled each grown-up and kid.
------------------------------------------------------------
[user] Now give me a recounting of our full conversation
------------------------------------------------------------
[assistant] Here’s a full recounting of our conversation so far:

1. You asked: “Please write a limerick about Stockholm”

2. I r

## Ownership is enforced

A different user (different `X-User-Id`) cannot read this conversation — the API returns 404, the same status it would for a non-existent one, so existence isn't leaked.

In [7]:
other = httpx.get(
    f"{BASE_URL}/api/v1/conversations/{session_id}/messages",
    headers={"X-User-Id": "someone-else"},
    timeout=10.0,
)
print(f"as 'someone-else' -> HTTP {other.status_code} (expected 404)")

as 'someone-else' -> HTTP 404 (expected 404)
